In [ ]:
from models.my_baseline_models import *

ModuleNotFoundError: No module named 'benchmark'

In [22]:
import pytorch_lightning as pl
from data_loader import SegmentationDataModule

class SegmentationTask(pl.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.loss_fn = nn.CrossEntropyLoss()

    def training_step(self, batch, batch_idx):
        x, y = batch
        # Your Dataset returns one-hot or 4D masks, convert to indices
        if y.ndim == 4: y = torch.argmax(y, dim=1)
        
        logits = self.model(x)
        loss = self.loss_fn(logits, y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)

# --- Execution ---
model_body = ResNet18UNet(in_channels=7, num_classes=4)
task = SegmentationTask(model_body)

# Setup DataModule
dm = SegmentationDataModule(root_dir='/local/s3167445/data', batch_size=16, num_workers=8)
dm.setup(stage='test')  # load test dataset

# Train on your RTX 3090s
trainer = pl.Trainer(
    accelerator="gpu", 
    devices=[0], # Use GPU 0 as identified in your nvidia-smi
    max_epochs=3,
    precision="16-mixed" # Utilize Tensor Cores on your 3090s for 2x speed
)

trainer.fit(task, datamodule=dm)
torch.save(model_body.state_dict(), "models/resnet18.pt")

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/local/s3167445/msc_venv/lib64/python3.9/site-packages/pytorch_lightning/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/local/s3167445/msc_venv/lib64/python3.9/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name    | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | model   | ResNet18UNet     | 13.8 M | train | 0    
1 | los

Epoch 2: 100%|██████████| 341/341 [00:19<00:00, 17.40it/s, v_num=5, train_loss=0.432]

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 2: 100%|██████████| 341/341 [00:25<00:00, 13.42it/s, v_num=5, train_loss=0.432]


In [ ]:
from torch.utils.data import DataLoader
from evaluation_functions import compute_miou
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_body.to(device)  
model_body.eval()    
temp_ds = dm.test_dataset
temp_loader = DataLoader(temp_ds, batch_size=2, num_workers=4)

current_miou = compute_miou(model_body, temp_loader,num_classes=4,device=device)
print(current_miou)


0.8767517805099487
